# Sugar Cosmetics - Instagram Performance Analysis (EDA)

This notebook walks through the full analysis step by step. Run the cells from top to bottom.

## Project links

| Item | Link |
| --- | --- |
| GitHub repository | ADD YOUR GITHUB LINK HERE |
| Streamlit live demo | ADD YOUR STREAMLIT LINK HERE |
| Tableau Public dashboard | ADD YOUR TABLEAU LINK HERE |
| Video explanation | ADD YOUR VIDEO LINK HERE |

## 1. Load the data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Pink colour theme and clean chart style
PINK = "#E6007E"
sns.set_theme(style="whitegrid")
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Read the cleaned file made by src/clean_data.py
df = pd.read_csv("../data/processed/sugar_cleaned.csv", parse_dates=["Post_Date"])

# Helper: save each chart into the images folder and show it
def save(name):
    plt.tight_layout()
    plt.savefig(f"../images/{name}.png", dpi=150)
    plt.show()

df.head()

## 2. Check the data quality

In [ ]:
# Size, missing values, duplicates and date range
print("Rows and columns:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Date range:", df["Post_Date"].min().date(), "to", df["Post_Date"].max().date())
print("Posts where reach is more than impressions:", (df["Reach"] > df["Impressions"]).sum())

## 3. Overview KPIs

Engagement rate = (likes + comments + shares + saves) / reach x 100

In [ ]:
# Main account numbers
kpi = {
    "Total posts": len(df),
    "Total likes": df["Likes"].sum(),
    "Total comments": df["Comments"].sum(),
    "Total reach": df["Reach"].sum(),
    "Total follows": df["Follows"].sum(),
    "Average engagement rate (%)": round(df["Engagement_Rate"].mean(), 2),
}
pd.Series(kpi).to_frame("Value")

## 4. Monthly posting trend

In [ ]:
# Count posts and average engagement for each month
monthly = df.groupby("Month_Year").agg(Posts=("Post_ID", "count"), Avg_ER=("Engagement_Rate", "mean")).reset_index()

plt.figure(figsize=(10, 4))
plt.plot(monthly["Month_Year"], monthly["Posts"], marker="o", color=PINK)
plt.title("Posts per month")
plt.xticks(rotation=45)
save("01_posts_per_month")

In [ ]:
# Average engagement rate by month
plt.figure(figsize=(10, 4))
plt.plot(monthly["Month_Year"], monthly["Avg_ER"], marker="o", color=PINK)
plt.ylim(17, 20)
plt.title("Average engagement rate by month (%)")
plt.xticks(rotation=45)
save("02_engagement_by_month")

## 5. Top 10 posts

In [ ]:
# Top 10 by likes
cols = ["Post_ID", "Post_Date", "Post_Type", "Campaign_Type", "Hashtags", "Likes", "Reach", "Engagement_Rate"]
top_likes = df.nlargest(10, "Likes")[cols]
top_likes

In [ ]:
# Top 10 by engagement rate
top_er = df.nlargest(10, "Engagement_Rate")[cols]
top_er

In [ ]:
# How many posts are in both top 10 lists?
print("Posts in both lists:", len(set(top_likes["Post_ID"]) & set(top_er["Post_ID"])))

plt.figure(figsize=(9, 4))
sns.barplot(data=top_likes, x="Likes", y=top_likes["Post_ID"].astype(str), color=PINK)
plt.title("Top 10 posts by likes")
plt.ylabel("Post ID")
save("03_top10_likes")

## 6. Post type comparison

In [ ]:
# Average likes, comments, saves and shares per post type
by_type = df.groupby("Post_Type")[["Likes", "Comments", "Saves", "Shares"]].mean().round(1)
by_type

In [ ]:
# Grouped bar chart (comments, saves, shares are small, so they are on their own chart)
by_type[["Comments", "Saves", "Shares"]].plot(kind="bar", figsize=(8, 4), colormap="RdPu")
plt.title("Average comments, saves and shares by post type")
plt.xticks(rotation=0)
save("04_post_type_metrics")

In [ ]:
# Engagement rate by post type
er_type = df.groupby("Post_Type")["Engagement_Rate"].mean().round(2).sort_values()
print(er_type)
plt.figure(figsize=(7, 4))
sns.barplot(x=er_type.index, y=er_type.values, color=PINK)
plt.ylim(17, 20)
plt.title("Average engagement rate by post type (%)")
save("05_er_post_type")

## 7. Day of the week

In [ ]:
# Average engagement rate for each weekday
by_day = df.groupby("Day_Name")["Engagement_Rate"].mean().reindex(DAY_ORDER).round(2)
print(by_day)
plt.figure(figsize=(9, 4))
sns.barplot(x=by_day.index, y=by_day.values, color=PINK)
plt.ylim(17, 20)
plt.title("Average engagement rate by weekday (%)")
save("06_er_weekday")

## 8. Engagement funnel

In [ ]:
# Total at every step: Impressions > Reach > Likes > Comments > Shares > Saves
steps = ["Impressions", "Reach", "Likes", "Comments", "Shares", "Saves"]
totals = df[steps].sum()
funnel = pd.DataFrame({"Total": totals, "Percent of previous step": (totals / totals.shift(1) * 100).round(1)})
funnel

In [ ]:
# Funnel-style horizontal bars (in millions)
plt.figure(figsize=(9, 4))
plt.barh(steps[::-1], (totals[steps] / 1e6).values[::-1], color=PINK)
plt.title("Engagement funnel (millions)")
save("07_funnel")

In [ ]:
# What happens after people see a post: profile visits and follows
print("Profile visits as % of reach:", round(df["Profile_Visits"].sum() / df["Reach"].sum() * 100, 2))
print("Follows as % of profile visits:", round(df["Follows"].sum() / df["Profile_Visits"].sum() * 100, 2))

## 9. Hashtag analysis

In [ ]:
# One row per hashtag
tags = df[["Hashtags", "Engagement_Rate", "Reach"]].copy()
tags["Hashtag"] = tags["Hashtags"].str.split(", ")
tags = tags.explode("Hashtag")

tag_sum = tags.groupby("Hashtag").agg(Uses=("Hashtag", "count"), Avg_ER=("Engagement_Rate", "mean"), Avg_Reach=("Reach", "mean")).round(2)
tag_sum

In [ ]:
# Engagement rate by hashtag
plt.figure(figsize=(9, 4))
sns.barplot(x=tag_sum.index, y=tag_sum["Avg_ER"], color=PINK)
plt.ylim(17, 20)
plt.title("Average engagement rate by hashtag (%)")
save("08_er_hashtag")

In [ ]:
# Does using more hashtags help?
by_count = df.groupby("Hashtag_Count")[["Engagement_Rate", "Reach", "Follows"]].mean().round(2)
by_count

## 10. Campaign type, audience emotion and engagement source

In [ ]:
# Three small tables side by side
for col in ["Campaign_Type", "Audience_Emotion", "Engagement_Source"]:
    print(df.groupby(col)[["Engagement_Rate", "Reach", "Follows"]].mean().round(2).sort_values("Engagement_Rate", ascending=False))
    print()

In [ ]:
# Chart: engagement rate by campaign type
camp = df.groupby("Campaign_Type")["Engagement_Rate"].mean().round(2).sort_values()
plt.figure(figsize=(8, 4))
sns.barplot(x=camp.values, y=camp.index, color=PINK)
plt.xlim(17, 20)
plt.title("Average engagement rate by campaign type (%)")
save("09_er_campaign")

## 11. Follower growth

In [ ]:
# New follows and follow rate for every full month (first and last month are partial)
growth = df.groupby("Month_Year").agg(Follows=("Follows", "sum"), Profile_Visits=("Profile_Visits", "sum"), Posts=("Post_ID", "count")).reset_index()
growth["Follow_Rate_%"] = (growth["Follows"] / growth["Profile_Visits"] * 100).round(1)

plt.figure(figsize=(10, 4))
plt.bar(growth["Month_Year"], growth["Follows"] / 1000, color=PINK)
plt.title("New follows per month (thousands)")
plt.xticks(rotation=45)
save("10_follows_per_month")
growth

## 12. Is any difference real?

The gaps between groups look very small. Here we compare the gap between groups with the normal spread of single posts.

In [ ]:
# Gap between best and worst group, compared with the spread of all posts
groups = ["Post_Type", "Day_Name", "Campaign_Type", "Audience_Emotion", "Engagement_Source"]
rows = []
for g in groups:
    means = df.groupby(g)["Engagement_Rate"].mean()
    rows.append({"Group": g, "Best": means.idxmax(), "Worst": means.idxmin(), "Gap (points)": round(means.max() - means.min(), 2)})
print("Spread of single posts (std):", round(df["Engagement_Rate"].std(), 2), "points")
pd.DataFrame(rows)

## 13. Conclusions

- The account is very even: every post type, weekday, hashtag and campaign has a similar engagement rate (about 18.5%).
- The small winners are Reels (post type), Wednesday (day), #VeganBeauty (hashtag) and Influencer Collab (campaign), but the gaps are tiny.
- Posts with 2 hashtags did slightly better than posts with 1 hashtag, and reach grows a little with more hashtags.
- Only about 3% of reached people visit the profile, and about 30% of profile visitors follow. New follows stay flat at about 700-760 thousand per month, so growth is steady but not rising. This is the best place to grow.